# Использование MLflow Tracking для экспериментов
MLflow Tracking позволяет логировать параметры, метрики и артефакты экспериментов. Рассмотрим, как начать с ним работать. Мы хотим добавить MLflow в существующий эксперимент и увидеть, что он может дать при работе с моделью.

In [ ]:
!pip install pandas scikit-learn

**Установка MLflow**

In [2]:
!pip install mlflow

   ---------------------------------------- 0.0/26.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/26.7 MB ? eta -:--:--
   -- ------------------------------------- 1.6/26.7 MB 5.2 MB/s eta 0:00:05
   ----- ---------------------------------- 3.9/26.7 MB 7.6 MB/s eta 0:00:04
   ------- -------------------------------- 5.0/26.7 MB 7.0 MB/s eta 0:00:04
   --------- ------------------------------ 6.3/26.7 MB 6.8 MB/s eta 0:00:04
   ----------- ---------------------------- 7.6/26.7 MB 6.5 MB/s eta 0:00:03
   ------------ --------------------------- 8.7/26.7 MB 6.5 MB/s eta 0:00:03
   -------------- ------------------------- 10.0/26.7 MB 6.4 MB/s eta 0:00:03
   ---------------- ----------------------- 11.3/26.7 MB 6.3 MB/s eta 0:00:03
   ------------------ --------------------- 12.6/26.7 MB 6.4 MB/s eta 0:00:03
   -------------------- ------------------- 13.9/26.7 MB 6.4 MB/s eta 0:00:03
   ---------------------- ----------------- 15.2/26.7 MB 6.4 MB/s eta 0:00:02
   -----

**Изначальный скрипт**

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

# Load the data
data = pd.read_csv('titanic.csv')

# Select features and target
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = data[features]
y = data['Survived']

# Preprocess the data
X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Scale the features
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.80


C:\Users\User\AppData\Local\Temp\ipykernel_4108\2678007223.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})


**Просмотр результатов в MLflow**

In [5]:
!mlflow ui

^C


# Использование MLflow Projects для воспроизведения экспериментов
MLflow Projects позволяет упаковать ваши проекты в стандартизированный формат, который легко воспроизводить. Это очень удобно, когда вы передаёте готовый проект заказчику или, например, когда у вас в компании много ML-проектов: разобраться в новом, когда они все устроены и организованы одинаково, сильно проще, чем когда они все разные.

**Структура проекта с MLflow Project**

my_project/
├── MLproject
├── conda.yaml
└── train.py

**Конфигурация файла MLproject**

In [7]:
name: My Project
conda_env: conda.yaml
entry_points:
  main:
    command: "python train.py"

SyntaxError: invalid syntax (2635282967.py, line 1)

**Создание окружения**

In [8]:
name: my_project_env
channels:
  - defaults
dependencies:
  - python=3.8
  - scikit-learn
  - pip
  - pip:
      - mlflow

SyntaxError: invalid syntax (2024842019.py, line 2)

**Скрипт для обучения модели**

In [9]:
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Set the MLflow tracking URI (optional, use if you have a specific server)
# mlflow.set_tracking_uri("http://your-mlflow-server:5000")

# Start an MLflow run
with mlflow.start_run():
    # Load the data
    data = pd.read_csv('titanic.csv')

    # Select features and target
    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
    X = data[features]
    y = data['Survived']

    # Preprocess the data
    X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})

    # Impute missing values
    imputer = SimpleImputer(strategy='mean')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    # Scale the features
    scaler = StandardScaler()
    X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train the model
    model = LogisticRegression()
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Log parameters
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("features", features)
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    # Log the model
    mlflow.sklearn.log_model(model, "model")

    # Print results
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")

    # Get the run ID
    run_id = mlflow.active_run().info.run_id
    print(f"MLflow Run ID: {run_id}")

C:\Users\User\AppData\Local\Temp\ipykernel_4108\1276186213.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})
2024/10/30 09:40:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


MlflowException: When an mlflow-artifacts URI was supplied, the tracking URI must be a valid http or https URI, but it was currently set to file:///C:/Users/User/YandexDisk/DS/MLAdvanced/11.%20%D0%9F%D1%80%D0%BE%D0%B4%D0%B2%D0%B8%D0%BD%D1%83%D1%82%D1%8B%D0%B5%20%D0%B8%D0%BD%D1%81%D1%82%D1%80%D1%83%D0%BC%D0%B5%D0%BD%D1%82%D1%8B%20ML-%D0%B8%D0%BD%D0%B6%D0%B5%D0%BD%D0%B5%D1%80%D0%B0/11.5/mlruns. Perhaps you forgot to set the tracking URI to the running MLflow server. To set the tracking URI, use either of the following methods:
1. Set the MLFLOW_TRACKING_URI environment variable to the desired tracking URI. `export MLFLOW_TRACKING_URI=http://localhost:5000`
2. Set the tracking URI programmatically by calling `mlflow.set_tracking_uri`. `mlflow.set_tracking_uri('http://localhost:5000')`

**Запуск MLflow Project**

In [ ]:
!mlflow run . -P n_estimators=100 -P max_depth=5

**Что происходит внутри**

После запуска выполняются важные шаги:
 1. **Чтение конфигурационного файла `MLproject`.**
MLflow начинает с чтения файла `MLproject`, который находится в корневой директории вашего проекта. Этот файл содержит метаданные проекта, включая информацию о зависимостях, команду для выполнения и другие настройки.
MLflow считывает имя проекта, описание окружения Conda и команду, которую нужно выполнить.

2. **Создание окружения.**
MLflow создаёт новое изолированное окружение Conda на основе файла `conda.yaml`. Этот файл определяет зависимости проекта, включая версии Python и необходимые библиотеки.

3. **Запуск команды.**
После настройки окружения MLflow выполняет команду, указанную в конфигурационном файле MLproject. В нашем примере команда запускает скрипт `train.py` с переданными параметрами `n_estimators` и `max_depth`.

4. **Логирование эксперимента.**
Когда скрипт `train.py` выполняется, он начинает новый эксперимент с помощью `mlflow.start_run()`.

5. **Сохранение артефактов.**
Все артефакты, включая модель и логированные метрики, сохраняются в локальном хранилище MLflow. Это позволяет в будущем легко воспроизвести эксперимент и проанализировать его результаты.

6. **Обновление интерфейса MLflow.**
После завершения выполнения скрипта все данные об эксперименте становятся доступными через интерфейс MLflow. Запустив MLflow UI командой mlflow ui, вы можете перейти в браузер по адресу `http://localhost:5000` и увидеть результаты эксперимента. В интерфейсе отображаются все логированные параметры, метрики и артефакты. Это позволяет удобно сравнивать разные эксперименты и анализировать их результаты.
